<a href="https://colab.research.google.com/github/huwperkins08-lang/TSF_seepage_detection_ElSoldado/blob/main/REP_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import Libraries, Retrive Data and Map Generation
##apply filters and set study area
###calculate REP

In [ ]:
import ee
import geemap.core as geemap
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import geemap
import matplotlib.patches as mpatches
import numpy as np
!pip install pymannkendall
import pymannkendall as mk
!pip install pingouin
import pingouin as pg

# 0. INITIALIZE GEE

# 1. INITIALIZATION
ee.Authenticate()
ee.Initialize(project='sxe390-el-soldado')

# 2. PARAMETERS & GEOMETRY
lat, lon = -32.65, -71.16
mine_site = ee.Geometry.Point([lon, lat])
study_area = mine_site.buffer(20000)

# 3. HELPER FUNCTIONS
#Cloud mask to remove cloudy pixels without descarding tile
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

#Aggregation of 10m spatial band resolution to 20m resolution of REP bands
def resample_to_20m(image):
    bands_10m = ['B2', 'B3', 'B4', 'B8']
    bands_20m = ['B5', 'B6', 'B7', 'B8A', 'B11', 'B12']
    target_projection = image.select('B5').projection()
    downsampled = (image.select(bands_10m)
                   .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
                   .reproject(crs=target_projection))
    return image.addBands(downsampled, overwrite=True).select(bands_10m + bands_20m)

#mask for steep terrain
def apply_terrain_mask(image):
    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    slope = ee.Terrain.slope(dem)
    return image.updateMask(slope.lte(20)) # Mask slopes > 20 degrees

#mask for illumination
def apply_illumination_mask(image):
    # 1. Safely get Sun angles. If they are missing, 'zen_val' becomes None
    zen_val = image.get('MEAN_SOLAR_ZENITH_ANGLE')
    azi_val = image.get('MEAN_SOLAR_AZIMUTH_ANGLE')

    # 2. Use a conditional: If the metadata exists, do the math. If not, return the original image.
    # This prevents the 'null' multiplication error.
    return ee.Image(ee.Algorithms.If(
        zen_val, # The condition: 'Does this value exist?'
        # TRUE CASE: Do the illumination maths
        (lambda: (
            image.updateMask(
                # Maths moved inside here to stay safe
                (terrain_math(image, zen_val, azi_val))
            )
        ))(),
        # FALSE CASE: Just return the image without the illumination mask
        image
    ))

# This is a helper function to keep the logic clean inside the 'If' statement
def terrain_math(image, zen, azi):
    degree2rad = 3.14159 / 180.0 #converts degrees to raidans
    z = ee.Number(zen).multiply(degree2rad)
    a = ee.Number(azi).multiply(degree2rad)

    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    terrain = ee.Terrain.products(dem)
    s = terrain.select('slope').multiply(degree2rad)
    asp = terrain.select('aspect').multiply(degree2rad)

    term1 = s.cos().multiply(z.cos())
    term2 = s.sin().multiply(z.sin()).multiply(asp.subtract(a).cos())
    cos_i = term1.add(term2)
    return cos_i.gt(0.1)#mask pixels under 0.1 illumination

def calculate_rep(image):# calculates REP using native Sentinel-2 bands and Guyot&Baret formula
    b4, b5, b6, b7 = image.select('B4'), image.select('B5'), image.select('B6'), image.select('B7')
    re_reflectance = (b4.add(b7)).divide(2)
    rep = ee.Image(705).add(
        ee.Image(35).multiply(
            re_reflectance.subtract(b5).divide(b6.subtract(b5))
        )
    ).rename('REP')
    #NDVI mask set to >0.3 to remove soil/background noise and isolate vegetation
    ndvi = image.normalizedDifference(['B8', 'B4'])
    return image.addBands(rep).updateMask(ndvi.gt(0.3))# values greater than 0.3NDVI included

# 4. SOURCE & FILTER
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(mine_site)
                  .filterDate('2017-01-01', '2026-05-31')
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))#removes tiles with over 40% cloud cover
                  .filter(ee.Filter.notNull(['MEAN_SOLAR_ZENITH_ANGLE', 'MEAN_SOLAR_AZIMUTH_ANGLE'])))

# 5. EXECUTION PIPELINE
processed_collection = (s2_collection
                        .map(mask_s2_clouds)
                        .map(resample_to_20m)
                        .map(apply_terrain_mask)
                        .map(apply_illumination_mask)
                        .map(calculate_rep))

# 6. GENERATE OUTPUT
final_median = processed_collection.median().clip(study_area)

# 7. VISUALIZATION
Map = geemap.Map(center=[lat, lon], zoom=13)
Map.addLayer(final_median, {'bands':['B4','B3','B2'], 'min':0, 'max':0.25}, 'Natural Color')

rep_params = {'bands':['REP'], 'min': 705, 'max': 720, 'palette': ['red', 'orange', 'yellow', 'green']}
Map.addLayer(final_median, rep_params, 'Red Edge Position (REP)')

Map

#Import Test and Cotrol sites from QGIS
##calculate pixels extracted

In [ ]:

# 1. Load  QGIS GeoJSON
gdf = gpd.read_file('sites_final_300m.geojson')

# 2. Convert to Earth Engine FeatureCollection
def geodf_to_ee(gdf):
    features = []
    for _, row in gdf.iterrows():
        # Convert QGIS geometry to EE geometry using the full GeoJSON interface
        # This handles various geometry types (Polygon, MultiPolygon, etc.) robustly.
        ee_geometry = ee.Geometry(row.geometry.__geo_interface__)
        f = ee.Feature(ee_geometry, row.drop('geometry').to_dict())
        features.append(f)
    return ee.FeatureCollection(features)

study_sites = geodf_to_ee(gdf)

# 3. Extract EVERY pixel value (REP) for each site
# We use 'final_median' which is your clean, 20m, terrain-masked composite
pixel_data = final_median.sampleRegions(
    collection=study_sites,
    properties=['site_type', 'site_name'],
    scale=20, # Matches our resampled resolution
    geometries=True
)

# 4. Convert to DataFrame
# Updated to select 'REP' (the name used in our calculate_rep function)
df_pixels = geemap.ee_to_df(pixel_data.select(['REP', 'site_type']))

print(f"Successfully extracted {len(df_pixels)} individual pixel values!")
print(df_pixels.head())

#Print Pixels per Site and Mean REP values

In [ ]:
# This will show exactly how many pixels were found for each site type
print("Breakdown of pixels by Site Type:")
print(df_pixels['site_type'].value_counts())

# This will show the average REP value for Test vs Control
print("\nMean REP values:")
print(df_pixels.groupby('site_type')['REP'].mean())


#Generate KDE plot
##add each control site and mean test site line

In [ ]:

# 1. Prepare the Data - we ensure 'Test' is the first color (Purple)
site_names = ['Test', 'Control_1', 'Control_2', 'Control_3', 'Control_4', 'Control_5']
colors = sns.color_palette('viridis', n_colors=len(site_names))

plt.figure(figsize=(12, 7))

# 2. Draw the curves - we use the exact same order here
sns.kdeplot(data=df_pixels, x='REP', hue='site_type', hue_order=site_names,
            fill=True, common_norm=False, palette='viridis', alpha=0.3)

# 3. Add the Mean Line
test_mean = df_pixels[df_pixels['site_type'] == 'Test']['REP'].mean()
mean_line = plt.axvline(test_mean, color='red', linestyle='--', label=f'Test Mean: {test_mean:.2f}')

# 4. Create the Legend handles manually to match the order
legend_handles = []
for i, name in enumerate(site_names):
    patch = mpatches.Patch(color=colors[i], label=name, alpha=0.5)
    legend_handles.append(patch)
legend_handles.append(mean_line)

# 5. Final Plot Touches
plt.legend(handles=legend_handles, title="Study Sites", loc='upper left', frameon=True)
plt.title('Red Edge Position (REP) Distribution: El Soldado Analysis', fontsize=16)
plt.xlabel('Red Edge Position (nm)', fontsize=13)
plt.ylabel('Pixel Density', fontsize=13)

plt.show()

#No. of satellite observations per site

In [ ]:

# 1. DEFINE EXTRACTION FUNCTION
def extract_test_site_stats(image):
    # Get the date
    date = image.date().format('YYYY-MM-dd')

    # Isolate just the 'Test' site from your GeoJSON FeatureCollection
    test_geom = study_sites.filter(ee.Filter.eq('site_type', 'Test')).geometry()

    # Calculate Mean REP for this specific image over the Test site
    stats = image.select('REP').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=test_geom,
        scale=20,
        maxPixels=1e9
    )

    # Return a Feature with the data
    return ee.Feature(None, {
        'date': date,
        'mean_rep': stats.get('REP')
    })

# 2. RUN THE HARVEST
# Note the spelling: FeatureCollection (no 'd')
temporal_fc = ee.FeatureCollection(processed_collection.map(extract_test_site_stats))

# 3. CONVERT TO DATAFRAME
df_temporal = geemap.ee_to_df(temporal_fc)

# 4. CLEANUP
df_temporal['date'] = pd.to_datetime(df_temporal['date'])
df_temporal = df_temporal.dropna().sort_values('date')

print(f"Captured {len(df_temporal)} individual satellite observations.")
df_temporal.head()

#Plot Test site observations

In [ ]:


# 1. Set the visual style
plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid")

# 2. Plot the raw observations
sns.scatterplot(data=df_temporal, x='date', y='mean_rep',
                color='purple', s=80, alpha=0.6, label='Individual Observation')

# 3. Add a trend line (3-point rolling average)
# This helps smooth out minor atmospheric variations
df_temporal['rolling_trend'] = df_temporal['mean_rep'].rolling(window=3, center=True).mean()
plt.plot(df_temporal['date'], df_temporal['rolling_trend'],
         color='purple', linewidth=3, label='Temporal Trend')

# 4. Add the "Health Threshold" reference
# Based on your KDE plot, the controls peaked around 720-721nm
plt.axhline(y=720.5, color='green', linestyle='--', linewidth=1.5, label='Healthy Baseline (Control Mean)')

# 5. Formatting
plt.title('2023 Red Edge Position (REP) Timeline: El Soldado Test Site', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('REP Wavelength (nm)', fontsize=12)
plt.ylim(715, 725) # Focuses the zoom on the shift
plt.legend()
plt.xticks(rotation=45)

plt.show()

#Plot test against mean controll sites

In [ ]:
# 1. New Extraction Function for ALL sites
def extract_all_sites_stats(image):
    date = image.date().format('YYYY-MM-dd')

    # We loop through each site type in your FeatureCollection
    def get_site_mean(site_type):
        geom = study_sites.filter(ee.Filter.eq('site_type', site_type)).geometry()
        return image.select('REP').reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=20
        ).get('REP')

    return ee.Feature(None, {
        'date': date,
        'Test': get_site_mean('Test'),
        'Control_1': get_site_mean('Control_1'),
        'Control_2': get_site_mean('Control_2'),
        'Control_3': get_site_mean('Control_3'),
        'Control_4': get_site_mean('Control_4'),
        'Control_5': get_site_mean('Control_5')
    })

# 2. HARVEST
full_temporal_fc = ee.FeatureCollection(processed_collection.map(extract_all_sites_stats))
df_full = geemap.ee_to_df(full_temporal_fc)

# 3. CLEANUP
df_full['date'] = pd.to_datetime(df_full['date'])

# NEW: Calculate the average of controls first
control_cols = ['Control_1', 'Control_2', 'Control_3', 'Control_4', 'Control_5']
df_full['Control_Avg'] = df_full[control_cols].mean(axis=1)

# NEW: Filter out the 'Spike' (Keep only realistic REP values)
df_clean = df_full[
    (df_full['Test'] > 700) & (df_full['Test'] < 730) &
    (df_full['Control_Avg'] > 700) & (df_full['Control_Avg'] < 730)
].dropna().sort_values('date')

# 4. PLOT COMPARISON (Using the cleaned data)
plt.figure(figsize=(15, 8))

# Plot Test Site
sns.lineplot(data=df_clean, x='date', y='Test', color='purple', linewidth=2.5, label='TEST SITE (Stream/TSF)')

# Plot Control Average
sns.lineplot(data=df_clean, x='date', y='Control_Avg', color='green', linewidth=2, linestyle='--', label='CONTROLS (Regional Average)')

plt.title('2018-2025 REP Comparison: Impact vs. Regional Baseline', fontsize=16)
plt.ylabel('Red Edge Position (nm)')
plt.ylim(705, 725) # This forces the view to the relevant spectral range
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"Removed outliers. Plotting {len(df_clean)} valid temporal observations.")

#Smoothed graph test vs control

In [ ]:

# 1. Calculate a 6-month rolling average to smooth the 8-year noise
df_clean['Test_Smooth'] = df_clean['Test'].rolling(window=12, center=True).mean()
df_clean['Control_Smooth'] = df_clean['Control_Avg'].rolling(window=12, center=True).mean()

# 2. Plot the smoothed trends
plt.figure(figsize=(15, 8))

plt.plot(df_clean['date'], df_clean['Test_Smooth'], color='purple', linewidth=3, label='Test Site (Smoothed)')
plt.plot(df_clean['date'], df_clean['Control_Smooth'], color='green', linewidth=3, linestyle='--', label='Controls (Smoothed)')

# 3. Fill the gap - this visually highlights the 'Stress Deficit'
plt.fill_between(df_clean['date'], df_clean['Test_Smooth'], df_clean['Control_Smooth'],
                 color='gray', alpha=0.2, label='Stress Gap')

plt.title('8-Year Vegetation Health Deficit: El Soldado (2018-2025)', fontsize=16)
plt.ylabel('Red Edge Position (nm)')
plt.ylim(714, 722)
plt.legend()
plt.show()

#Calculate average gap wet and dry season

In [ ]:
# 1. Create a column for the 'Gap' (Control minus Test)
df_clean['Stress_Gap'] = df_clean['Control_Avg'] - df_clean['Test']

# 2. Compare the Gap in 'Dry' vs 'Wet' seasons (Approximate for Chile)
# Wet Season: June, July, August (Months 6, 7, 8)
df_clean['month'] = df_clean['date'].dt.month
df_clean['season'] = df_clean['month'].apply(lambda x: 'Wet' if x in [6, 7, 8] else 'Dry')

seasonal_gap = df_clean.groupby('season')['Stress_Gap'].mean()

print("--- Seasonal Stress Analysis ---")
print(f"Average Gap (Dry Season): {seasonal_gap['Dry']:.2f} nm")
print(f"Average Gap (Wet Season): {seasonal_gap['Wet']:.2f} nm")

# 3. Correlation Check
correlation = df_clean['Test'].corr(df_clean['Control_Avg'])
print(f"\nOverall 8-Year Correlation: {correlation:.3f}")

#Monthly average Test vs Control

In [ ]:
# 1. Create a "Climatology" of the Red Edge Position
# We group by month to see the average seasonal cycle over the 8 years
monthly_climatology = df_clean.groupby('month')[['Test', 'Control_Avg']].mean().reset_index()

# 2. Plot the seasonal cycle
plt.figure(figsize=(12, 6))

plt.plot(monthly_climatology['month'], monthly_climatology['Control_Avg'],
         marker='o', color='green', linewidth=3, label='Controls (Normal Cycle)')

plt.plot(monthly_climatology['month'], monthly_climatology['Test'],
         marker='o', color='purple', linewidth=3, label='Test Site (Impacted Cycle)')

# 3. Highlight the Winter Gap (June-August)
plt.axvspan(6, 8, color='blue', alpha=0.1, label='Wet Season (Contaminant Flush)')

plt.title('The Seasonal Inversion: Average REP Cycle (2018-2025)', fontsize=15)
plt.xlabel('Month of Year', fontsize=12)
plt.ylabel('Mean Red Edge Position (nm)', fontsize=12)
plt.xticks(range(1, 13))
plt.grid(axis='y', alpha=0.3)
plt.legend()

plt.show()

#List monthly average test vs controll and gap

In [ ]:
# Create a summary of the Gap by Month
summary_table = monthly_climatology.copy()
summary_table['Gap'] = summary_table['Control_Avg'] - summary_table['Test']
summary_table.columns = ['Month', 'Test REP (nm)', 'Control Avg (nm)', 'Stress Gap (nm)']

print("--- FINAL MONTHLY STRESS SUMMARY (2018-2025) ---")
print(summary_table.to_string(index=False))

# Retrive Climate data for El Soldado Region using CHIRPS

In [ ]:

# 1. Define the Date Range
years = ee.List.sequence(2017, 2025)
months = ee.List.sequence(1, 12)

# 2. Load the CHIRPS collection
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

# 3. Function to calculate Monthly Sums
def get_monthly_precip(year):
    def month_map(month):
        # Filter for the specific year and month
        monthly_data = (chirps
                        .filter(ee.Filter.calendarRange(year, year, 'year'))
                        .filter(ee.Filter.calendarRange(month, month, 'month')))

        # Sum the daily rainfall
        monthly_sum = monthly_data.sum()

        # Calculate the mean rainfall across your study area
        stats = monthly_sum.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=mine_site.buffer(5000), # 5km area around El Soldado
            scale=5000
        )

        return ee.Feature(None, {
            'precipitation': stats.get('precipitation'),
            'year': year,
            'month': month,
            # 'system:time_start': ee.Date.fromYMD(year, month, 1).millis() # No longer needed here if constructing date from year/month columns
        })
    return months.map(month_map)

# 4. Execute and Flatten
monthly_precip_features = ee.FeatureCollection(years.map(get_monthly_precip).flatten())

# 5. Convert to Pandas DataFrame
df_rain = geemap.ee_to_df(monthly_precip_features)
# Corrected: Create 'date' column from 'year' and 'month' columns
df_rain['date'] = pd.to_datetime(df_rain[['year', 'month']].assign(day=1))

print(df_rain.head())


# Create monthly averages for rainfall

In [ ]:
df_month_average = df_rain.groupby('month')['precipitation'].mean().reset_index()
df_month_average
print(df_month_average.head())

# Plot monthly rainfall averages for region against monthy REP averages

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

# 1. Plot the REP data on the LEFT axis
ax1.plot(monthly_climatology['month'], monthly_climatology['Control_Avg'],
         marker='o', color='green', linewidth=3, label='Controls (Normal Cycle)')

ax1.plot(monthly_climatology['month'], monthly_climatology['Test'],
         marker='o', color='purple', linewidth=3, label='Test Site (Impacted Cycle)')

ax1.set_ylabel('Mean Red Edge Position (nm)', fontsize=12)
ax1.set_ylim(715, 721) # Focus the view on the actual REP range

# 2. Create the second axis for Rainfall on the RIGHT
ax2 = ax1.twinx()
ax2.bar(df_month_average['month'], df_month_average['precipitation'],
        color='blue', alpha=0.3, label='CHIRPS Rainfall (mm)')

ax2.set_ylabel('Mean Rainfall (mm)', color='blue', fontsize=12)
ax2.set_ylim(0, df_month_average['precipitation'].max() * 1.5) # Scale to rainfall

# 3. Aesthetics
plt.axvspan(5, 8, color='blue', alpha=0.2, label='Wet Season (Contaminant Flush)')
ax1.set_title('The Seasonal Inversion: REP vs Rainfall (2018-2025)', fontsize=15)
ax1.set_xlabel('Month of Year', fontsize=12)
ax1.set_xticks(range(1, 13))

# Combine legends from both axes
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper left')

plt.show()


#Create Control site dataframe for temporal analysis of REP using Seasonal Mann Kendall & Sen test

In [ ]:
#cretae dataframe for temporal analysis of REP at test site
df_SMK_stats = df_temporal.resample('M', on='date').median()

#count number of NaN returned in mean_rep column
print(f"Number of NaN values in 'mean_rep' column: {df_SMK_stats['mean_rep'].isna().sum()}")

#identify NaN months in dataframe
nan_dates = df_SMK_stats[df_SMK_stats['mean_rep'].isna()].index
nan_year_month = [(d.year, d.month) for d in nan_dates]
print(f"Year-Months with NaN values in 'mean_rep': {nan_year_month}")

#replace anomolous REP values with NaN
df_SMK_stats.loc[df_SMK_stats['mean_rep'] < 700, 'mean_rep'] = np.nan
df_SMK_stats.loc[df_SMK_stats['mean_rep'] > 730, 'mean_rep'] = np.nan

#print 5 smallest and largest values to check
print(df_SMK_stats['mean_rep'].nsmallest(5))
print(df_SMK_stats['mean_rep'].nlargest(5))

In [ ]:
#plot seasonal trend in REP
plt.figure(figsize=(12, 6))
plt.plot(df_SMK_stats.index, df_SMK_stats['mean_rep'], marker='o', linestyle='-', color='green')
plt.title('Monthly Mean REP Over Time', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Mean REP (nm)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Plot Autocorrelation Function to test for correlated values and apply appropriate version of Seasonal Mann-Kendall & Sen test

In [ ]:

from statsmodels.graphics.tsaplots import plot_acf

# 1. Prepare data by dropping the NaN values so the calculation can run
acf_data = df_SMK_stats['mean_rep'].dropna()

# 2. Plot the Autocorrelation Function
fig, ax = plt.subplots(figsize=(10, 5))

# lags=24 looks back over a 2-year window (24 months) to capture seasonal cycles
plot_acf(acf_data, lags=24, ax=ax)

ax.set_title("Autocorrelation Function (ACF) - Monthly Test Site REP", fontsize=14)
ax.set_xlabel("Lag (Months)", fontsize=12)
ax.set_ylabel("Autocorrelation Coefficient", fontsize=12)

plt.tight_layout()
plt.show()

Use Correlation adjusted Seasonal Mann-Kendall & Sen test

In [ ]:

import pymannkendall as mk

# ─── 1. ALIGN DATAFRAME TO THE PERFECT 113-MONTH TIMELINE ─────────────
# Ensure index is a datetime index and sorted chronologically
df_SMK_stats.index = pd.to_datetime(df_SMK_stats.index)
df_SMK_stats = df_SMK_stats.sort_index()

# Create a continuous monthly template from Jan 2017 to May 2026
# Changed freq from 'MS' (Month Start) to 'ME' (Month End) to match df_SMK_stats index
perfect_timeline = pd.date_range(start="2017-01-01", end="2026-05-31", freq="ME")

# Reindex matches dates and automatically fills missing gaps with np.nan
df_processed = df_SMK_stats.reindex(perfect_timeline)

# ─── 2. ISOLATE THE DESEASONALISED RESIDUALS ───────────────────────────────
# We calculate and subtract the seasonal mean for each month grouping
historical_monthly_means = df_processed.groupby(df_processed.index.month)['mean_rep'].transform('mean')
df_processed['deseasonalised_rep'] = df_processed['mean_rep'] - historical_monthly_means

# ─── 3. RUN THE CORRECT AUTOCORRELATION-MITIGATED SEASONAL TEST ────────────
# Convert the structured column straight into a NumPy flat matrix
rep_array = df_processed['mean_rep'].to_numpy()
mkmk_results = mk.correlated_seasonal_test(rep_array, period=12)

# ─── 4. GENERATE THE DUAL-PANEL PLOT ───────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Top Plot: Raw Sentinel-2 Red Edge Position
ax1.plot(df_processed.index, df_processed['mean_rep'], marker='o', markersize=4, linestyle='-', color='#2ca02c', label='Raw REP')
ax1.set_title("Sentinel-2 Red Edge Position Time Series (2017 - 2026)", fontsize=12, fontweight='bold')
ax1.set_ylabel("Red Edge Position (nm)")
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper left')

# Bottom Plot: Deseasonalised Trend Analysis
ax2.plot(df_processed.index, df_processed['deseasonalised_rep'], marker='s', markersize=4, linestyle='--', color='#1f77b4', label='Deseasonalised Residuals')

# CORRECTED SEN'S SLOPE RENDERING:
# Create an integer array representing months elapsed from the start of the study
months_elapsed = np.arange(len(df_processed))
# Calculate the midpoint of the timeline to center our zero-mean trend line
midpoint = np.mean(months_elapsed)

# Generate the trend line equation: slope * (x - midpoint)
# This anchors the line perfectly at 0 right in the center of the timeline
trend_line = mkmk_results.slope * (months_elapsed - midpoint)

# Plot the red trend line on the deseasonalised axis
ax2.plot(df_processed.index, trend_line, color='red', linewidth=2,
         label=f"Sen's Slope: {mkmk_results.slope:.4f}/mo (p={mkmk_results.p:.3f})")

ax2.set_title(f"Underlying Long-Term Trend (Detected: {mkmk_results.trend.upper()})", fontsize=12, fontweight='bold')
ax2.set_xlabel("Timeline")
ax2.set_ylabel("Deviation from Seasonal Mean")
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

# ─── 5. PRINT STATISTICS SUMMARY ───────────────────────────────────────────
print("="*45)
print("  CORRECTED SEASONAL MANN-KENDALL RESULTS  ")
print("="*45)
print(f"Trend Evaluation:       {mkmk_results.trend}")
print(f"Adjusted P-value:       {mkmk_results.p:.6f}")
print(f"True Sen's Slope:       {mkmk_results.slope:.6f} nm/month")
print("="*45)



In [ ]:
#create dataframe with median REP values for all sites for RM ANOVA
#create median monthly values for each site

df_ANOVA=df_full.resample('M', on='date').median()
#remove Control_Avg column to avoid confusion
df_ANOVA=df_ANOVA.drop(columns=['Control_Avg'])
# Count and locate number of NaN entries
print(f"Number of NaN values:{df_ANOVA.isna().sum()}")
df_NaN_months=df_ANOVA[df_ANOVA.isna().any(axis=1)]
print(f"NaN months:{df_NaN_months.index}")

#drop entire rows where NaN values exist
df_ANOVA=df_ANOVA.dropna()
print(f"Number of NaN values:{df_ANOVA.isna().sum()}")

#drop entire rows where REP values fall outside the range
valid_range_mask = ((df_ANOVA>=700) & (df_ANOVA<=730)).all(axis=1)
df_ANOVA=df_ANOVA[valid_range_mask]

#count number of rows in df
print(f"Number of rows:{len(df_ANOVA)}")

# Reset the index to make 'date' a regular column before melting
df_ANOVA = df_ANOVA.reset_index()

#Sort df for RM ANOVA
df_ANOVA=df_ANOVA.melt(
    id_vars=['date'],
    value_vars=['Test', 'Control_1', 'Control_2', 'Control_3', 'Control_4', 'Control_5'],
    var_name='Site',
    value_name='REP'

)

df_ANOVA['Group'] = df_ANOVA['Site'].apply(lambda x: 'Test' if x == 'Test' else 'Control')

df_ANOVA = df_ANOVA.sort_values(by=['date', 'Site']).reset_index(drop=True)
df_ANOVA

In [ ]:
#find the 5 smallest values in datafram
df_ANOVA.nsmallest(5, 'REP')

In [ ]:
#run shapiro wilk test on dataframe
from scipy.stats import shapiro
df_ANOVA_rep=df_ANOVA['REP']
stat, p = shapiro(df_ANOVA_rep)
print('Statistics=%.3f, p=%.3f' % (stat, p))

In [ ]:
# Execute Mauchly's test of sphericity
spher_results = pg.sphericity(
    data=df_ANOVA, dv="REP", within="Site", subject="date", method="mauchly"
    )
print(spher_results)

In [ ]:
#plot scattergraph of REP values for all sites
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_ANOVA, x='date', y='REP', hue='Site')
plt.title('REP Values for All Sites Over Time', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('REP (nm)', fontsize=12)

In [ ]:
from statsmodels.stats.anova import AnovaRM

# Run the Repeated Measures ANOVA across all 6 individual sites
# within=['Site_ID'] tests the main effect of the 6 separate locations
anova_site_model = AnovaRM(
    data=df_ANOVA,
    depvar='REP',
    subject='date',
    within=['Site']
)
anova_site_results = anova_site_model.fit()

print("="*55)
print("   REPEATED MEASURES ANOVA: INDIVIDUAL SITE VARIANCE   ")
print("="*55)
print(anova_site_results)
print("="*55)

In [ ]:
# --- TEST 1: THE BASELINE ERA (Pre-2020) ---
df_baseline = df_ANOVA[df_ANOVA['date'].dt.year < 2020].copy()

baseline_anova = AnovaRM(
    data=df_baseline,
    depvar='REP',
    subject='date',
    within=['Site']
).fit()
print("=== BASELINE ERA RESULTS (2017-2019) ===")
print(baseline_anova)

# --- TEST 2: THE POST-EXPANSION ERA (2020-2026) ---
df_post = df_ANOVA[df_ANOVA['date'].dt.year >= 2020].copy()

post_anova = AnovaRM(
    data=df_post,
    depvar='REP',
    subject='date',
    within=['Site']
).fit()
print("\n=== POST-EXPANSION ERA RESULTS (2020-2026) ===")
print(post_anova)

In [ ]:
# 1. Create a fresh copy from df_clean
df_arima = df_clean.copy()

# 2. Make sure Python recognizes the 'date' column as actual datetime objects
df_arima['date'] = pd.to_datetime(df_arima['date'])

# 3. Set 'date' as the index, then select the numeric columns you want to resample,
#    and then resample to a monthly end frequency (ME), taking the median.
#    This aggregates multiple observations within a month to a single monthly value.
df_arima = df_arima.set_index('date')[['Test', 'Control_Avg']].resample('ME').median()

# 4. Drop any rows where these key columns are NaN (can result from months with no observations)
df_arima = df_arima.dropna()

# 5. Create environmental driver (The Wetter Season Proxy)
df_arima['Is_Wet_Season'] = df_arima.index.month.isin([6, 7, 8]).astype(int)

df_arima

In [ ]:
#plot new arima dataframe for visual analysis
plt.figure(figsize=(12, 6))
# Plot 'Test' series
plt.plot(df_arima.index, df_arima['Test'], marker='o', linestyle='-', color='blue', label='Test Site')
# Plot 'Control_Avg' series
plt.plot(df_arima.index, df_arima['Control_Avg'], marker='o', linestyle='-', color='green', label='Control Average')
plt.title('Monthly Median REP Over Time', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Median REP (nm)', fontsize=12)
plt.legend() # Add legend to differentiate the lines
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.api as sm

# 1. Define the dependent variable (y) and independent variables (X)
y = df_arima['Test']
X = df_arima[['Control_Avg', 'Is_Wet_Season']]

# 2. Add a constant term so the model can calculate a baseline intercept
X = sm.add_constant(X)

# 3. Fit the Autoregressive Error Model (AR1)
# order=(1,0,0) specifies a 1-month autoregressive lag to absorb temporal memory
arima_model = sm.tsa.statespace.SARIMAX(y, exog=X, order=(1, 0, 0))
arima_results = arima_model.fit(disp=False)

# 4. Display the results
print(arima_results.summary())

In [ ]:

# CHIRPS RAIN DF - change data format to month end
# The 'date' column was already set as the index. To convert the month-start index to month-end:
df_rain.index = df_rain.index + pd.offsets.MonthEnd(0)

# Display the modified DataFrame
df_rain

In [ ]:
df_lagged = df_arima.copy()
#add monthly precipitation from CHIRPS to lagged df
df_lagged['Precipitation'] = df_rain['precipitation']
#remove NaN from precipitation column
df_lagged = df_lagged.dropna()
#remove wet season column
df_lagged = df_lagged.drop(columns=['Is_Wet_Season'])
#Create lagged columns
df_lagged['Lag_0']=df_lagged['Precipitation']
df_lagged['Lag_1']=df_lagged['Precipitation'].shift(1)
df_lagged['Lag_2']=df_lagged['Precipitation'].shift(2)
df_lagged['Lag_3']=df_lagged['Precipitation'].shift(3)
#drop all rows with NaN values
df_lagged=df_lagged.dropna()
df_lagged

In [ ]:
#plot test column and lag columns against date
fig, ax1 = plt.subplots(figsize=(12, 6))

# Calculate rolling means for smoothing
window_size = 3 # 3-month rolling mean
df_lagged['Test_smoothed'] = df_lagged['Test'].rolling(window=window_size, center=True).mean()

# Smooth the lagged precipitation columns as well
for col in ['Lag_0', 'Lag_1', 'Lag_2', 'Lag_3']:
    df_lagged[f'{col}_smoothed'] = df_lagged[col].rolling(window=window_size, center=True).mean()

# Plot smoothed 'Test' series
ax1.plot(df_lagged.index, df_lagged['Test_smoothed'], marker='o', linestyle='-', color='blue', label='Test Site (Smoothed)')

# Plot smoothed lagged precipitation columns
ax2 = ax1.twinx()
ax2.plot(df_lagged.index, df_lagged[['Lag_0_smoothed', 'Lag_1_smoothed', 'Lag_2_smoothed', 'Lag_3_smoothed']], marker='o', linestyle='-', color='green', label='Lagged Columns (Smoothed)')

ax1.set_title('Monthly Median REP Over Time (Smoothed)', fontsize=14)
ax1.set_xlabel('Date', fontsize=12)
ax1.set_ylabel('Median REP (nm)', fontsize=12)
ax2.set_ylabel('Smoothed Precipitation (mm)', fontsize=12)
ax1.legend()
ax2.legend()
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_xticks(ax1.get_xticks(), labels=ax1.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Target variable stays the same
y = df_lagged['Test']

# 2. Define your new predictors (Control baseline + the explicit rain lags)
X_lags = df_lagged[['Control_Avg', 'Lag_0', 'Lag_1', 'Lag_2', 'Lag_3']]
X_lags = sm.add_constant(X_lags)

# 3. Fit the SARIMAX model, keeping the 1-month autoregressive memory filter
lag_model = sm.tsa.statespace.SARIMAX(y, exog=X_lags, order=(1, 0, 0))
lag_results = lag_model.fit(disp=False)

# 4. Print the new results
print(lag_results.summary())